# Project 14 — Handwritten Digit Classifier (MNIST)

This notebook walks through the full pipeline:
1. Load & explore the MNIST dataset
2. Build and train a neural network
3. Evaluate performance
4. Save the trained model
5. Run inference on custom handwritten digit images

It reuses the exact same code as `src/train.py` and `src/predict.py` so results here match what the scripts produce.

In [ ]:
import sys
from pathlib import Path

# Allow imports from the project's src/ package when running from notebooks/
ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import MODEL_PATH, CUSTOM_DIGITS_DIR, REPORTS_DIR, EPOCHS, VALIDATION_SPLIT
from src.model import build_model
from src.preprocessing import preprocess_image_path

print("TensorFlow version:", tf.__version__)

## 1. Load and explore the data

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0

print("Train shape:", x_train.shape, y_train.shape)
print("Test shape:", x_test.shape, y_test.shape)

In [ ]:
# Visualize a few sample digits
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(str(y_train[i]))
    ax.axis("off")
plt.suptitle("Sample MNIST training digits")
plt.show()

## 2. Build the model

Architecture (see `src/model.py`):
- Flatten 28x28 -> 784
- Dense(128, relu) + Dropout(0.2)
- Dense(64, relu)
- Dense(10, softmax)

In [ ]:
model = build_model()
model.summary()

## 3. Train the model

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

In [ ]:
# Look at a few test predictions, including any mistakes
preds = model.predict(x_test[:16])
pred_labels = preds.argmax(axis=1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(x_test[i], cmap="gray")
    correct = pred_labels[i] == y_test[i]
    color = "green" if correct else "red"
    ax.set_title(f"Pred: {pred_labels[i]} / True: {y_test[i]}", color=color, fontsize=9)
    ax.axis("off")
plt.suptitle("Test set predictions (green = correct, red = wrong)")
plt.tight_layout()
plt.show()

## 5. Save the trained model

In [ ]:
MODEL_PATH.parent.mkdir(exist_ok=True, parents=True)
model.save(MODEL_PATH)
print(f"Saved model to {MODEL_PATH}")

## 6. Run inference on custom handwritten digit images

Add your own `.png`/`.jpg` images of handwritten digits to `data/custom_digits/` and re-run this cell.
Preprocessing (grayscale, resize, auto-invert, normalize) is handled by `src/preprocessing.py` — the same code the standalone `src/predict.py` script and the Streamlit app use, so results are consistent everywhere.

In [ ]:
image_paths = sorted(list(CUSTOM_DIGITS_DIR.glob("*.png")) + list(CUSTOM_DIGITS_DIR.glob("*.jpg")))

if not image_paths:
    print(f"No custom images found in {CUSTOM_DIGITS_DIR}.")
    print("Add some handwritten digit images (.png/.jpg) there and re-run this cell.")
else:
    n = len(image_paths)
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).flatten() if n > 1 else [axes]

    for ax, img_path in zip(axes, image_paths):
        img_array = preprocess_image_path(img_path)
        pred = model.predict(img_array, verbose=0)
        digit = int(pred.argmax())
        confidence = float(pred.max())
        print(f"{img_path.name}: predicted={digit} confidence={confidence:.2%}")

        ax.imshow(img_array.reshape(28, 28), cmap="gray")
        ax.set_title(f"{img_path.name}\nPred: {digit} ({confidence:.0%})")
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Summary

- Trained a fully connected neural network on MNIST
- Achieved strong test-set accuracy (see printed value above)
- Saved the model to `models/mnist_digit_model.h5`
- Verified inference works on both MNIST test images and custom handwritten images
- The same model powers `src/predict.py` (CLI) and `app/streamlit_app.py` (web UI)